# Benchmarking Tabular Explainer
To benchmark `shapiq`'s `TabularExplainer` we run it against the equivalent explainers of the shap library. For comparing different approximators and imputer strategies We run both, the `shapiq` and the `shap` versions against the exact calculation of shapley values.

## Options for Implementation / Options to be Varied

### Approximators

`shapiq`                | number of runs         | Imputers / Masker     | Komment
------------------------|------------------------|-----------------------|------------------------
KernelSHAP              | baseline -> don't care | -                     | exact calculation, no approx
SVARM                   | no information found   | marginal, conditional, baseline | special shapiq thing - what to compare with?
PermutationSamplingSV   |                        |                       |

`shap`                  | number of runs         | Imputers / Masker     | Komment
------------------------|------------------------|-----------------------|------------------------
KernelExplainer         | no information found   | kind of anything      | maybe shap.maskers.impute?
PermutationExplainer    | no information found   | shap.maskers.impute?  |

### Imputer

Mind sample_size and conditional_budget to get the same number of runs with all imputers

##### shapiq:

marginal, conditional, baseline, gaussian conditional imputer, gaussian copula conditional imputer

##### shap:



In [ ]:
from __future__ import annotations

import numpy as np
import shapiq
from shapiq.games.benchmark.setup import GameBenchmarkSetup
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

import shapiq_student

## Preparing Datasets and models
We use the `california_housing` and the `bike_sharing` datasets for the benchmark as they are comparably small datasets (so the exact shapley values can be computed in reasonable time) still they are realistic (real world) data sets.
The values scaled to a range from -1 to 1 and split up in training and test data (80:20).
As both data sets have non-classified output, we use regression models to be explained in the benchmark. To verify, if the explainer performance is depending on the model to be explained we use two different models from the scikit-learn library: `KNeighborsRegressor` and `DecisionTreeRegressor`.

In [2]:
bike_tree = GameBenchmarkSetup(dataset_name="bike_sharing", model_name="decision_tree")
a = bike_tree.n_features
b = bike_tree.fit_function  # returns a callable

house_tree = GameBenchmarkSetup(dataset_name="california_housing", model_name="decision_tree")
c = house_tree.n_features
d = house_tree.fit_function

bike = shapiq.datasets.load_bike_sharing(to_numpy=True)
house = shapiq.datasets.load_california_housing(to_numpy=True)

bike_x_train, bike_x_test, bike_y_train, bike_y_test = train_test_split(
    bike[0], bike[1], test_size=0.05, random_state=42
)
house_x_train, house_x_test, house_y_train, house_y_test = train_test_split(
    house[0], house[1], test_size=0.05, random_state=42
)

tree_bike_model = DecisionTreeRegressor()

k_neighbor_bike = KNeighborsRegressor(10)
k_neighbor_bike.fit(bike_x_train, bike_y_train)

k_neighbor_house = KNeighborsRegressor(10)
k_neighbor_house.fit(bike_x_train, bike_y_train)


# display(bike_tree.x_data)
# display(bike_tree.x_train)

# display(bike_tree.x_test)

# display(bike_tree.y_data)
# display(bike_tree.y_train)

# display(bike_tree.y_test)

# display(bike_x_train)
# display(house_x_test)

# display(bike_y_test)
# display(house_y_test)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


##Imputation
As the performance concerning speed as well as quality of an approximator depends strongly on the used imputer, we are going to use different imputers for comparison.
On the other hand the performance of an approximator depends on the number of passes so we are going to run the test on the same amount for each approximator.


In [ ]:
import shapiq_student.explainer
import shapiq_student.explainer.knn

sample_size = (
    100  # The number of samples to draw from the conditional background data for the imputation.
)

conditional_budget = (
    10,
    100,
    10,
)  # The number of coalitions to sample per each point in `data` for training the generative model.
conditional_threshold = (
    10  # Quantile threshold defining a neighbourhood of samples to draw `sample_size` from.
)
random_state = 42

kern_house = shapiq.approximator.KernelSHAP(len(house_x_train[0]))
kern_bike = shapiq.approximator.KernelSHAP(len(bike_x_train[0]))

svarm_house = shapiq.approximator.SVARM(n=len(house_x_train[0]), index="SV", random_state=42)
svarm_bike = shapiq.approximator.SVARM(n=len(bike_x_train[0]), index="SV", random_state=42)

for i in range(10):
    house_tree_kernel = shapiq.TabularExplainer(
        model=house_tree.model,
        data=house_tree.x_train,
        class_index=None,
        imputer="marginal",
        approximator=kern_house,
        index="SV",
        max_order=1,
        random_state=42,
        verbose=True,
        sample_size=sample_size,
        joint_marginal_distribution=False,
        normalize=False,
    )
    house_tree_svarm = shapiq.TabularExplainer(
        model=house_tree.model,
        data=house_tree.x_train,
        class_index=None,
        imputer="marginal",
        approximator=svarm_house,
        index="SV",
        max_order=1,
        random_state=42,
        verbose=True,
        sample_size=sample_size,
        joint_marginal_distribution=False,
        normalize=False,
    )
    h_t_k_expl = np.array((len(house_x_train[0]), len(conditional_budget)))
    h_t_s_expl = np.array((len(house_x_train[0]), len(conditional_budget)))

    for j in range(len(conditional_budget)):
        h_t_k_expl[j] = shapiq_student.explainer.knn.base.interaction_values_to_array(
            house_tree_kernel.explain_function(
                x=house_tree.x_test[i], budget=conditional_budget[j], random_state=42
            )
        )
        # h_t_k_expl.add(t)
        # display(h_t_k_expl)
        # display(h_t_k_expl.values)
        h_t_s_expl[j] = shapiq_student.explainer.knn.base.interaction_values_to_array(
            house_tree_svarm.explain(
                x=house_tree.x_test[i], budget=conditional_budget[j], random_state=42
            )
        )
        # h_t_s_expl.add(s)
        # display(h_t_s_expl)
        # display(h_t_s_expl.values)
    #     h_t_k_seq = shapiq.interaction_values.InteractionValues.add(h_t_k_expl)
    #     h_t_s_seq = shapiq.interaction_values.InteractionValues.add(h_t_s_expl)

    # h_t_k_av = shapiq.interaction_values.aggregate_interaction_values(h_t_k_seq)
    # h_t_s_av = shapiq.interaction_values.aggregate_interaction_values(h_t_s_seq)

Evaluating game:   0%|          | 0/10 [00:00<?, ? coalition/s]

ValueError: setting an array element with a sequence.